# FULL TRANSFORMER ARCHITECTURE

In [2]:
import tensorflow as tf

In [3]:
tf.random.set_seed(42)

In [4]:
sentences = [
    "I love machine learning models",
    "Transformers understand human language well",
    "Deep learning models learn patterns",
    "Attention helps models understand context"
]

In [5]:
seq_len = 5
d_model = 8

In [6]:
import numpy as np
from tensorflow.keras.layers import TextVectorization, Embedding, MultiHeadAttention


In [7]:
tokenizer = TextVectorization(
    output_mode ="int",
    output_sequence_length = seq_len
)

tokenizer.adapt(sentences)

In [8]:
token_ids = tokenizer(tf.constant(sentences))

In [9]:
token_ids.shape

TensorShape([4, 5])

In [10]:
vocab_size = len(tokenizer.get_vocabulary())

In [11]:
print(vocab_size)

18


In [12]:
embedding = Embedding(
    input_dim = vocab_size,
    output_dim = d_model,
)

In [13]:
embeddings = embedding(token_ids)

In [14]:
print("\nEmbeddings shape:", embeddings.shape)


Embeddings shape: (4, 5, 8)


In [15]:
def positional_encoding(seq_len,d_model):
  pos = np.arange(seq_len)[:,np.newaxis]
  i = np.arange(d_model)[np.newaxis,:]
  angle_rate = 1/np.power(
      1000,
      (2*(i//1))/np.float32(d_model)
  )
  angle_rads = pos*angle_rate

  # sine
  angle_rads[:,0::2] = np.sin(
      angle_rads[:,0::2]
  )

  # cos
  angle_rads[:,1::2] = np.cos(
      angle_rads[:,1::2]
  )

  return tf.cast(angle_rads[np.newaxis,...],
                 dtype = tf.float32)


In [16]:
pos_encoding = positional_encoding(seq_len, d_model)

In [17]:
print("shape of positional encoding",pos_encoding.shape)

shape of positional encoding (1, 5, 8)


In [18]:
final_input = embeddings + pos_encoding

In [19]:
print("after add of embedding with positional encoding ",final_input.shape)

after add of embedding with positional encoding  (4, 5, 8)


In [20]:
multi_head = 2
key_dim = 4

In [21]:
multi_head_attention = MultiHeadAttention(
    num_heads = multi_head,
    key_dim = key_dim
)

In [22]:
output,attention_scores = multi_head_attention(
    query = final_input,
    value = final_input,
    key = final_input,
    return_attention_scores = True
)

In [23]:
print("output shape",output.shape)

output shape (4, 5, 8)


In [24]:
print("attention score shape",attention_scores.shape)

attention score shape (4, 2, 5, 5)


In [25]:
input_shape = final_input + output

In [26]:
print(input_shape.shape)

(4, 5, 8)


In [27]:
add_norm1 = tf.keras.layers.LayerNormalization()
add_norm1 = add_norm1(input_shape)

In [29]:
print(add_norm1.shape)

(4, 5, 8)


In [36]:
ffnn = tf.keras.Sequential([
    tf.keras.layers.Dense(128,activation = "relu"),
    tf.keras.layers.Dense(d_model)
])

In [37]:
ffnn_output = ffnn(add_norm1)

In [38]:
print(ffnn_output.shape)

(4, 5, 8)


In [39]:
add_norm2 = tf.keras.layers.LayerNormalization()
add_norm2 = add_norm2(add_norm1 + ffnn_output)

In [40]:
print(add_norm2.shape)

(4, 5, 8)
